# 🏦 PME Credit Scoring — Dual-Model Stacking Framework
**Dataset:** `pme_final_scored_scale10_data`  
**Target:** `target_is_solvable`  
**Architecture:** Base Model 1 (Financial/Operational) + Base Model 2 (Behavioral/Compliance) → Meta Logistic Regression

---

## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    RocCurveDisplay, ConfusionMatrixDisplay, roc_curve
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from scipy.stats import skew

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = {'0': '#e74c3c', '1': '#2ecc71'}
sns.set_palette(['#2ecc71', '#e74c3c'])
print('✅ Imports done')

## 1. Load Data

In [ ]:
df = pd.read_csv('data/pme_final_scored_scale10_data.csv')  # data file is in ./data
# If tab-separated:
# df = pd.read_csv('data/pme_final_scored_scale10_data.csv', sep='\t')

print(f'Shape: {df.shape}')
df.head(3)

## 2. Feature Groups & Column Definitions

In [ ]:
TARGET = 'target_is_solvable'

# ❌ Always drop
DROP_COLS = ['id', 'FinScore', 'Cluster_ID', 'Risk_Tier']

# 🟢 Model 1: Financial + Operational
M1_FEATURES = [
    'business_turnover_tnd',
    'business_expenses_tnd',
    'profit_margin',
    'nbr_of_workers',
    'workers_verified_cnss',
    'formal_worker_ratio',
    'business_age_years',
    'number_of_owners'
]

# 🔵 Model 2: Behavioral + Compliance
M2_FEATURES_NUM = [
    'compliance_rne_score',
    'steg_sonede_score',
    'banking_maturity_score',
    'followers_fcb',
    'followers_insta',
    'followers_linkedin',
    'posts_per_month'
]
M2_FEATURES_CAT = ['type_of_business']

print('🟢 M1 features:', M1_FEATURES)
print('🔵 M2 num features:', M2_FEATURES_NUM)
print('🔵 M2 cat features:', M2_FEATURES_CAT)

---
# 🔍 EDA

### A. Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df[TARGET].value_counts()
pct = df[TARGET].value_counts(normalize=True) * 100

# Bar
bars = axes[0].bar(
    ['Not Solvable (0)', 'Solvable (1)'],
    counts.values,
    color=['#e74c3c', '#2ecc71'], edgecolor='white', linewidth=1.5
)
for bar, val, p in zip(bars, counts.values, pct.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{val}\n({p:.1f}%)', ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Target Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')

# Pie
axes[1].pie(counts.values, labels=['Not Solvable', 'Solvable'],
            colors=['#e74c3c', '#2ecc71'], autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Proportion', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('eda_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

imbalance_ratio = counts.min() / counts.max()
print(f'\n⚖️  Imbalance ratio: {imbalance_ratio:.2f}')
if imbalance_ratio < 0.7:
    print('⚠️  Imbalance detected → will use class_weight="balanced" in models')
else:
    print('✅ Classes are reasonably balanced')

### B. Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if missing_df.empty:
    print('✅ No missing values found')
else:
    print(f'⚠️  {len(missing_df)} features with missing values:')
    display(missing_df.style.background_gradient(cmap='Reds', subset=['Missing %']))
    
    fig, ax = plt.subplots(figsize=(10, max(4, len(missing_df)*0.4)))
    ax.barh(missing_df.index, missing_df['Missing %'], color='#e74c3c')
    ax.set_xlabel('Missing %')
    ax.set_title('Missing Values by Feature', fontweight='bold')
    plt.tight_layout()
    plt.savefig('eda_missing_values.png', dpi=150, bbox_inches='tight')
    plt.show()

### C. Numerical Feature Distributions — Skewness & Outliers

In [ ]:
all_num_features = M1_FEATURES + M2_FEATURES_NUM

# Skewness table
skewness = df[all_num_features].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
skew_df = pd.DataFrame({'Skewness': skewness.round(2)})
skew_df['Flag'] = skew_df['Skewness'].apply(lambda s:
    '🔴 Highly Skewed' if abs(s) > 2 else ('🟡 Moderately Skewed' if abs(s) > 0.5 else '🟢 OK'))
print('Skewness Analysis:')
display(skew_df)

In [ ]:
# Distribution plots
n = len(all_num_features)
ncols = 3
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3.5))
axes = axes.flatten()

for i, feat in enumerate(all_num_features):
    color = '#27ae60' if feat in M1_FEATURES else '#2980b9'
    axes[i].hist(df[feat].dropna(), bins=40, color=color, alpha=0.8, edgecolor='white')
    sk = skew(df[feat].dropna())
    axes[i].set_title(f'{feat}\nskew={sk:.2f}', fontsize=9, fontweight='bold')
    axes[i].set_ylabel('Count', fontsize=8)
    group = '🟢 M1' if feat in M1_FEATURES else '🔵 M2'
    axes[i].text(0.98, 0.95, group, transform=axes[i].transAxes,
                 ha='right', va='top', fontsize=8,
                 bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.3))

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numerical Feature Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

### D. Feature Relationship with Target

In [ ]:
# 🟢 Model 1 features vs target
m1_key = ['profit_margin', 'formal_worker_ratio', 'business_turnover_tnd']

fig, axes = plt.subplots(1, len(m1_key), figsize=(15, 4))
for i, feat in enumerate(m1_key):
    df.boxplot(column=feat, by=TARGET, ax=axes[i],
               patch_artist=True,
               boxprops=dict(facecolor='#2ecc71', color='#27ae60'),
               medianprops=dict(color='white', linewidth=2))
    axes[i].set_title(feat, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('target_is_solvable (0=No, 1=Yes)')

plt.suptitle('🟢 Model 1: Financial Features vs Target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_m1_vs_target.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 🔵 Model 2 features vs target
m2_key = ['compliance_rne_score', 'banking_maturity_score', 'steg_sonede_score']

fig, axes = plt.subplots(1, len(m2_key), figsize=(15, 4))
for i, feat in enumerate(m2_key):
    df.boxplot(column=feat, by=TARGET, ax=axes[i],
               patch_artist=True,
               boxprops=dict(facecolor='#3498db', color='#2980b9'),
               medianprops=dict(color='white', linewidth=2))
    axes[i].set_title(feat, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('target_is_solvable (0=No, 1=Yes)')

plt.suptitle('🔵 Model 2: Behavioral Features vs Target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_m2_vs_target.png', dpi=150, bbox_inches='tight')
plt.show()

### E. Correlation Analysis (Within Each Model Group)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# M1 correlation
corr_m1 = df[M1_FEATURES].corr()
mask_m1 = np.triu(np.ones_like(corr_m1, dtype=bool))
sns.heatmap(corr_m1, ax=axes[0], mask=mask_m1, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.7})
axes[0].set_title('🟢 Model 1: Financial Correlations', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# M2 correlation
corr_m2 = df[M2_FEATURES_NUM].corr()
mask_m2 = np.triu(np.ones_like(corr_m2, dtype=bool))
sns.heatmap(corr_m2, ax=axes[1], mask=mask_m2, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.7})
axes[1].set_title('🔵 Model 2: Behavioral Correlations', fontsize=12, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('eda_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

# Flag high correlations
def high_corr_pairs(corr_matrix, threshold=0.8):
    pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            val = corr_matrix.iloc[i, j]
            if abs(val) >= threshold:
                pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], round(val, 3)))
    return pairs

print('\n🔴 High correlation pairs (|r| >= 0.8):')
for c1, c2, v in high_corr_pairs(corr_m1) + high_corr_pairs(corr_m2):
    print(f'   {c1} ↔ {c2}: {v}')
if not (high_corr_pairs(corr_m1) + high_corr_pairs(corr_m2)):
    print('   ✅ None found above threshold')

### F. Zero-Inflated Social Media Features

In [ ]:
social_features = ['followers_fcb', 'followers_insta', 'followers_linkedin', 'posts_per_month']

zero_pct = (df[social_features] == 0).mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Zero % bar chart
bars = axes[0].bar(social_features, zero_pct.values, color='#e67e22', edgecolor='white')
for bar, pct in zip(bars, zero_pct.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{pct:.1f}%', ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('Zero Values in Social Features (%)', fontweight='bold')
axes[0].set_ylabel('% Zero')
axes[0].set_ylim(0, 110)
axes[0].tick_params(axis='x', rotation=20)

# Distribution only for non-zero
feat = 'followers_fcb'
non_zero = df[df[feat] > 0][feat]
axes[1].hist(non_zero, bins=40, color='#e67e22', alpha=0.8, edgecolor='white')
axes[1].set_title(f'{feat} — Non-Zero Values Only\n(n={len(non_zero)})', fontweight='bold')
axes[1].set_xlabel('Followers')

plt.tight_layout()
plt.savefig('eda_zero_inflation.png', dpi=150, bbox_inches='tight')
plt.show()

print('💡 Insight: Missing social values should be replaced with 0 → means "no digital presence"')

---
# ⚙️ Preprocessing

### Step 1: Drop Leakage Columns

In [ ]:
df_clean = df.drop(columns=DROP_COLS, errors='ignore')
print(f'Dropped: {DROP_COLS}')
print(f'Remaining columns: {df_clean.shape[1]}')

### Step 2: Handle Missing Values

In [ ]:
# Social features: fill NaN with 0 (= no digital presence)
df_clean[social_features] = df_clean[social_features].fillna(0)

# Other numerics: fill with median
other_num = [f for f in M1_FEATURES + M2_FEATURES_NUM if f not in social_features]
for col in other_num:
    if col in df_clean.columns:
        median_val = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_val)

# Categorical: fill with mode
for col in M2_FEATURES_CAT:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

print(f'Remaining missing: {df_clean.isnull().sum().sum()}')
print('✅ Missing values handled')

### Step 3 & 4: Log Transformation for Skewed Financial Features

In [ ]:
LOG_FEATURES = ['business_turnover_tnd', 'business_expenses_tnd',
                'followers_fcb', 'followers_insta', 'followers_linkedin']

for feat in LOG_FEATURES:
    if feat in df_clean.columns:
        df_clean[f'{feat}_log'] = np.log1p(df_clean[feat])  # log1p handles zeros
        print(f'  log1p({feat}) → skew: {skew(df[feat].dropna()):.2f} → {skew(df_clean[feat+"_log"]):.2f}')

# Update M1 features to use log-transformed versions
M1_FEATURES_PROCESSED = [
    'business_turnover_tnd_log',
    'business_expenses_tnd_log',
    'profit_margin',
    'nbr_of_workers',
    'workers_verified_cnss',
    'formal_worker_ratio',
    'business_age_years',
    'number_of_owners'
]

M2_FEATURES_NUM_PROCESSED = [
    'compliance_rne_score',
    'steg_sonede_score',
    'banking_maturity_score',
    'followers_fcb_log',
    'followers_insta_log',
    'followers_linkedin_log',
    'posts_per_month'
]

print('\n✅ Log transformation applied')

### Step 5: Encode Categorical Feature

In [ ]:
# Fix encoding issues (e.g. Ã©)
df_clean['type_of_business'] = df_clean['type_of_business'].str.encode('latin1', errors='ignore').str.decode('utf-8', errors='ignore')

business_dummies = pd.get_dummies(df_clean['type_of_business'], prefix='biz', drop_first=False)
print('One-Hot Encoded categories:')
print(business_dummies.columns.tolist())

df_clean = pd.concat([df_clean, business_dummies], axis=1)

# Add dummy column names to M2 features
M2_FEATURES_ALL = M2_FEATURES_NUM_PROCESSED + business_dummies.columns.tolist()
print(f'\n✅ M2 total features after encoding: {len(M2_FEATURES_ALL)}')

### Step 7: Train / Test Split (Stratified)

In [ ]:
X = df_clean
y = df_clean[TARGET].astype(int)

# Stratified split: 80% train, 20% test
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size: {len(X_train_full)} | Test size: {len(X_test_full)}')
print(f'Train target distribution:\n{y_train.value_counts(normalize=True).round(3)}')
print(f'\nTest target distribution:\n{y_test.value_counts(normalize=True).round(3)}')

### Step 6 & 8: Build Base Model Pipelines with Scaling

In [ ]:
# Extract feature subsets
X_train_m1 = X_train_full[M1_FEATURES_PROCESSED]
X_test_m1  = X_test_full[M1_FEATURES_PROCESSED]

X_train_m2 = X_train_full[M2_FEATURES_ALL]
X_test_m2  = X_test_full[M2_FEATURES_ALL]

# Scale
scaler_m1 = StandardScaler()
scaler_m2 = StandardScaler()

X_train_m1_sc = scaler_m1.fit_transform(X_train_m1)
X_test_m1_sc  = scaler_m1.transform(X_test_m1)

X_train_m2_sc = scaler_m2.fit_transform(X_train_m2)
X_test_m2_sc  = scaler_m2.transform(X_test_m2)

print('✅ Features scaled (StandardScaler)')

---
# 🤖 Training Base Models

### 🟢 Model 1: Financial + Operational Risk (Gradient Boosting)

In [ ]:
model1 = GradientBoostingClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)
model1.fit(X_train_m1_sc, y_train)

# Cross-validation
cv_scores_m1 = cross_val_score(model1, X_train_m1_sc, y_train, cv=5, scoring='roc_auc')
print(f'🟢 Model 1 CV AUC: {cv_scores_m1.mean():.4f} ± {cv_scores_m1.std():.4f}')

# Test predictions
p1_train = model1.predict_proba(X_train_m1_sc)[:, 1]
p1_test  = model1.predict_proba(X_test_m1_sc)[:, 1]
print(f'🟢 Model 1 Test AUC: {roc_auc_score(y_test, p1_test):.4f}')

### 🔵 Model 2: Behavioral + Compliance Risk (Random Forest)

In [ ]:
model2 = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
model2.fit(X_train_m2_sc, y_train)

cv_scores_m2 = cross_val_score(model2, X_train_m2_sc, y_train, cv=5, scoring='roc_auc')
print(f'🔵 Model 2 CV AUC: {cv_scores_m2.mean():.4f} ± {cv_scores_m2.std():.4f}')

p2_train = model2.predict_proba(X_train_m2_sc)[:, 1]
p2_test  = model2.predict_proba(X_test_m2_sc)[:, 1]
print(f'🔵 Model 2 Test AUC: {roc_auc_score(y_test, p2_test):.4f}')

### Feature Importance: Model 1

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# M1 importance
imp_m1 = pd.Series(model1.feature_importances_, index=M1_FEATURES_PROCESSED).sort_values(ascending=True)
imp_m1.plot(kind='barh', ax=axes[0], color='#27ae60', edgecolor='white')
axes[0].set_title('🟢 Model 1: Feature Importances', fontweight='bold')
axes[0].set_xlabel('Importance')

# M2 importance
imp_m2 = pd.Series(model2.feature_importances_, index=M2_FEATURES_ALL).sort_values(ascending=True).tail(15)
imp_m2.plot(kind='barh', ax=axes[1], color='#2980b9', edgecolor='white')
axes[1].set_title('🔵 Model 2: Feature Importances (Top 15)', fontweight='bold')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('model_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
# 🔥 Stacking: Meta-Model (Logistic Regression)

### Step 9: Build Meta-Feature Dataset

In [ ]:
# Stack probabilities
X_meta_train = np.column_stack([p1_train, p2_train])
X_meta_test  = np.column_stack([p1_test,  p2_test])

print('Meta-feature dataset shape (train):', X_meta_train.shape)
print('Meta-feature dataset shape (test): ', X_meta_test.shape)
print('\nMeta features = [p1_financial, p2_behavioral]')

# Quick peek
meta_df_sample = pd.DataFrame(X_meta_train[:5], columns=['p1_financial', 'p2_behavioral'])
meta_df_sample['target'] = y_train.values[:5]
display(meta_df_sample)

### Step 10: Train Meta-Model

In [ ]:
meta_model = LogisticRegression(
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
meta_model.fit(X_meta_train, y_train)

# Meta-model coefficients (interpret weights)
coef_df = pd.DataFrame({
    'Feature': ['p1_financial', 'p2_behavioral'],
    'Coefficient': meta_model.coef_[0]
})
print('\n🧠 Meta-Model Coefficients (signal weights):')
display(coef_df)

y_pred_meta = meta_model.predict(X_meta_test)
p_meta_test = meta_model.predict_proba(X_meta_test)[:, 1]

print(f'\n🏆 Meta-Model Test AUC: {roc_auc_score(y_test, p_meta_test):.4f}')

### Step 11: Save Trained Artifacts for src Modules
Save models, scalers, processed data, and feature parameters so src modules can run without retraining in the same kernel.

In [ ]:
from pathlib import Path
import joblib

ARTIFACTS_DIR = Path('src/artifacts')
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

# Core models and scalers
joblib.dump(model1, ARTIFACTS_DIR / 'model1_gb.joblib')
joblib.dump(model2, ARTIFACTS_DIR / 'model2_rf.joblib')
joblib.dump(meta_model, ARTIFACTS_DIR / 'meta_model_lr.joblib')
joblib.dump(scaler_m1, ARTIFACTS_DIR / 'scaler_m1.joblib')
joblib.dump(scaler_m2, ARTIFACTS_DIR / 'scaler_m2.joblib')

# Data snapshots used by explainability (keeps index alignment by saving objects directly)
joblib.dump(df, ARTIFACTS_DIR / 'df_raw.joblib')
joblib.dump(df_clean, ARTIFACTS_DIR / 'df_clean.joblib')

# Training and feature configuration
training_params = {
    'target': TARGET,
    'drop_cols': DROP_COLS,
    'log_features': LOG_FEATURES,
    'm1_features': M1_FEATURES,
    'm2_features_num': M2_FEATURES_NUM,
    'm2_features_cat': M2_FEATURES_CAT,
    'm1_features_processed': M1_FEATURES_PROCESSED,
    'm2_features_num_processed': M2_FEATURES_NUM_PROCESSED,
    'm2_features_all': M2_FEATURES_ALL,
    'decision_thresholds': {
        'low_risk_min_prob': 0.67,
        'medium_risk_min_prob': 0.45
    }
}
joblib.dump(training_params, ARTIFACTS_DIR / 'training_params.joblib')

print(f'Artifacts saved in: {ARTIFACTS_DIR.resolve()}')
print('Saved files:')
for p in sorted(ARTIFACTS_DIR.glob('*')):
    print(f' - {p.name}')

---
# 📊 Final Evaluation

### Classification Reports

In [ ]:
for name, y_pred, y_prob in [
    ('🟢 Model 1 (Financial)', model1.predict(X_test_m1_sc), p1_test),
    ('🔵 Model 2 (Behavioral)', model2.predict(X_test_m2_sc), p2_test),
    ('🏆 Meta-Model (Stacked)', y_pred_meta, p_meta_test)
]:
    auc = roc_auc_score(y_test, y_prob)
    print(f'\n{name}  |  AUC: {auc:.4f}')
    print('-' * 55)
    print(classification_report(y_test, y_pred, target_names=['Not Solvable', 'Solvable']))

### ROC Curve Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# ROC curves
ax = axes[0]
for name, prob, color in [
    ('Model 1 – Financial', p1_test, '#27ae60'),
    ('Model 2 – Behavioral', p2_test, '#2980b9'),
    ('Meta-Model (Stacked)', p_meta_test, '#e74c3c')
]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_val = roc_auc_score(y_test, prob)
    ax.plot(fpr, tpr, color=color, linewidth=2.5, label=f'{name} (AUC={auc_val:.3f})')

ax.plot([0,1],[0,1], 'k--', linewidth=1, alpha=0.5, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curve Comparison', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Confusion matrix for meta model
ax2 = axes[1]
cm = confusion_matrix(y_test, y_pred_meta)
ConfusionMatrixDisplay(cm, display_labels=['Not Solvable', 'Solvable']).plot(ax=ax2, cmap='Blues', colorbar=False)
ax2.set_title('🏆 Meta-Model Confusion Matrix', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

### Probability Distribution: Meta-Model

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for cls, color, label in [(0, '#e74c3c', 'Not Solvable'), (1, '#2ecc71', 'Solvable')]:
    probs = p_meta_test[y_test == cls]
    ax.hist(probs, bins=30, alpha=0.65, color=color, label=f'{label} (n={len(probs)})',
            edgecolor='white', density=True)

ax.axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='Decision Threshold (0.5)')
ax.set_xlabel('Predicted Probability of Solvable', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('🏆 Meta-Model: Predicted Probability Distribution', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('meta_probability_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### Model Comparison Summary

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

results = []
for name, y_pred, y_prob in [
    ('Model 1 – Financial', model1.predict(X_test_m1_sc), p1_test),
    ('Model 2 – Behavioral', model2.predict(X_test_m2_sc), p2_test),
    ('Meta-Model (Stacked)', y_pred_meta, p_meta_test)
]:
    results.append({
        'Model': name,
        'AUC': round(roc_auc_score(y_test, y_prob), 4),
        'F1': round(f1_score(y_test, y_pred, average='weighted'), 4),
        'Precision': round(precision_score(y_test, y_pred, average='weighted'), 4),
        'Recall': round(recall_score(y_test, y_pred, average='weighted'), 4)
    })

results_df = pd.DataFrame(results).set_index('Model')
display(results_df.style
        .background_gradient(cmap='Greens', subset=['AUC', 'F1'])
        .format('{:.4f}'))

---
# 🧠 Summary

| Layer | Model | Signal |
|-------|-------|--------|
| Base 🟢 | Gradient Boosting | Is the business **financially viable**? |
| Base 🔵 | Random Forest | Is the business **trustworthy / structured**? |
| Meta 🏆 | Logistic Regression | How to **combine both signals** optimally? |

**Key design decisions:**
- Log-transform for skewed financial features
- Zero-fill for social features (= no digital presence)
- `class_weight='balanced'` to handle any class imbalance
- Stratified split to preserve class proportions
- Meta-model uses only `[p1, p2]` → minimal risk of overfitting

---
## 9. Explainability (SHAP)
This section is executed after training and stacking are complete. It explains one company prediction using SHAP for each base model, then aggregates strengths and weaknesses.

In [ ]:
import sys
from pathlib import Path
import joblib

SRC_DIR = Path('src').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from explainability import ExplainabilityEngine

def build_explainability_engine():
    required_names = [
        'df', 'df_clean', 'model1', 'model2', 'meta_model',
        'scaler_m1', 'scaler_m2', 'M1_FEATURES_PROCESSED', 'M2_FEATURES_ALL'
    ]
    missing = [name for name in required_names if name not in globals()]
    if missing:
        raise ValueError('Missing training artifacts. Run previous training cells first: ' + ', '.join(missing))

    return ExplainabilityEngine(
        df=df,
        df_clean=df_clean,
        model1=model1,
        model2=model2,
        meta_model=meta_model,
        scaler_m1=scaler_m1,
        scaler_m2=scaler_m2,
        m1_features_processed=M1_FEATURES_PROCESSED,
        m2_features_all=M2_FEATURES_ALL,
    )

def load_explainability_engine_from_artifacts(artifacts_dir='src/artifacts'):
    artifacts = Path(artifacts_dir)
    model1 = joblib.load(artifacts / 'model1_gb.joblib')
    model2 = joblib.load(artifacts / 'model2_rf.joblib')
    meta_model = joblib.load(artifacts / 'meta_model_lr.joblib')
    scaler_m1 = joblib.load(artifacts / 'scaler_m1.joblib')
    scaler_m2 = joblib.load(artifacts / 'scaler_m2.joblib')
    df_raw = joblib.load(artifacts / 'df_raw.joblib')
    df_clean_loaded = joblib.load(artifacts / 'df_clean.joblib')
    params = joblib.load(artifacts / 'training_params.joblib')

    return ExplainabilityEngine(
        df=df_raw,
        df_clean=df_clean_loaded,
        model1=model1,
        model2=model2,
        meta_model=meta_model,
        scaler_m1=scaler_m1,
        scaler_m2=scaler_m2,
        m1_features_processed=params['m1_features_processed'],
        m2_features_all=params['m2_features_all'],
    )

print('Explainability module ready.')
print('- Use build_explainability_engine() in the same training session.')
print('- Use load_explainability_engine_from_artifacts() after kernel restart.')

In [ ]:
# Example usage after training cells are executed:
# explainer = build_explainability_engine()
# company_result = explainer.explain(company_id='PME_000023', top_n=5)
# company_result['shap_full_df'].head(10)

## 10. Report Generation
Use the external modules to generate the full PDF report (with SHAP waterfall/bar plots) from the explainability output.

In [ ]:
from pipeline import AnalysisPipeline
from report_generator import ReportGenerator

def build_pipeline(output_dir='reports', max_display=10):
    explainer = build_explainability_engine()
    report_gen = ReportGenerator(output_dir=output_dir, max_display=max_display)
    return AnalysisPipeline(explainer=explainer, report_generator=report_gen)

def load_pipeline_from_artifacts(artifacts_dir='src/artifacts', output_dir='reports', max_display=10):
    explainer = load_explainability_engine_from_artifacts(artifacts_dir=artifacts_dir)
    report_gen = ReportGenerator(output_dir=output_dir, max_display=max_display)
    return AnalysisPipeline(explainer=explainer, report_generator=report_gen)

print('Pipeline module ready.')
print('- Use build_pipeline() in current session.')
print('- Use load_pipeline_from_artifacts() from saved files.')

In [ ]:
# Example workflow (same session after training):
# pipeline = build_pipeline(output_dir='reports', max_display=10)
# run_result = pipeline.analyze_and_report(company_id='PME_000023', top_n=5, logo_path=None)
# print(run_result['pdf_path'])

# Example workflow (new session / after restart):
# pipeline = load_pipeline_from_artifacts(artifacts_dir='src/artifacts', output_dir='reports', max_display=10)
# run_result = pipeline.analyze_and_report(company_id='PME_000023', top_n=5, logo_path=None)
# print(run_result['pdf_path'])